In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns

# Read the CSV file
df = pd.read_csv("C:/Users/M S I/Downloads/OneDrive_2025-09-18/Predicting Olympic medal counts/data/olympics2016.csv")

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import glm, poisson, negativebinomial
from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# Read the CSV file
df = pd.read_csv("C:/Users/M S I/Downloads/OneDrive_2025-09-18/Predicting Olympic medal counts/data/olympics2016.csv")

# Data preparation
print("Data Preparation...")
print("="*50)

# Identify potential predictors (excluding identifiers and redundant columns)
potential_predictors = [
    'gdp00', 'gdp04', 'gdp08', 'gdp12', 'gdp16', 'pop00', 'pop04', 'pop08', 'pop12', 'pop16', 
    'soviet', 'comm', 'muslim', 'oneparty', 'gold00', 'gold04', 'gold08', 'gold12', 'tot16', 
    'tot00', 'tot04', 'tot08', 'tot12', 'totgold00', 'totgold04', 'totgold08', 'totgold12', 'totgold16', 
    'totmedals00', 'totmedals04', 'totmedals08', 'totmedals12', 'totmedals16', 'bmi', 'altitude', 
    'athletes00', 'athletes04', 'athletes08', 'athletes12', 'athletes16', 'host'
]

# Identify countries with missing gdp16 and gdp00
missing_gdp16 = df[df['gdp16'].isnull()]['country'].tolist()
missing_gdp00 = df[df['gdp00'].isnull()]['country'].tolist()

print(f"Countries with missing gdp16: {missing_gdp16}")
print(f"Countries with missing gdp00: {missing_gdp00}")

# Select relevant numeric variables for KNN imputation
# Choose variables that are correlated with GDP
predictor_columns = ['gdp12', 'gdp08', 'gdp04', 'gdp00', 'gdp16',  # Include both target variables
                    'pop16', 'pop12', 'pop08', 'pop04', 'pop00', 
                    'soviet', 'comm', 'muslim', 'oneparty']

# Filter to only include columns that exist in the dataset
available_predictors = [col for col in predictor_columns if col in df.columns]
print(f"Using predictors: {available_predictors}")

# Create a subset with both GDP variables and predictors
# We'll impute gdp00 and gdp16 together since they're related
impute_data = df[['gdp00', 'gdp16'] + [p for p in available_predictors if p not in ['gdp00', 'gdp16']]].copy()

print(f"\nData for imputation shape: {impute_data.shape}")
print(f"Missing values before imputation:")
print(f"gdp00: {impute_data['gdp00'].isnull().sum()}")
print(f"gdp16: {impute_data['gdp16'].isnull().sum()}")

# Standardize the data (important for KNN)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(impute_data)

# Apply KNN imputation
imputer = KNNImputer(n_neighbors=5)
imputed_data = imputer.fit_transform(scaled_data)

# Convert back to original scale
imputed_data_original = scaler.inverse_transform(imputed_data)

# Create DataFrame with imputed values
imputed_df = pd.DataFrame(imputed_data_original, 
                         columns=impute_data.columns,
                         index=df.index)

# Get the imputed GDP values for the missing countries
imputed_gdp00_values = imputed_df.loc[df['gdp00'].isnull(), 'gdp00']
imputed_gdp16_values = imputed_df.loc[df['gdp16'].isnull(), 'gdp16']

print("\n" + "=" * 50)
print("KNN IMPUTATION RESULTS")
print("=" * 50)

print("\nGDP00 Imputation Results:")
for i, (idx, value) in enumerate(imputed_gdp00_values.items()):
    country = df.loc[idx, 'country']
    original_value = df.loc[idx, 'gdp00']
    print(f"{country}: {value:,.0f} (imputed)")

print("\nGDP16 Imputation Results:")
for i, (idx, value) in enumerate(imputed_gdp16_values.items()):
    country = df.loc[idx, 'country']
    original_value = df.loc[idx, 'gdp16']
    print(f"{country}: {value:,.0f} (imputed)")

# ADD THE IMPUTED VALUES DIRECTLY TO ORIGINAL COLUMNS (NO NEW COLUMNS)
print("\n" + "=" * 50)
print("UPDATING ORIGINAL COLUMNS")
print("=" * 50)

# Store original missing counts
missing_gdp00_before = df['gdp00'].isnull().sum()
missing_gdp16_before = df['gdp16'].isnull().sum()

# Update the original columns with imputed values
df.loc[df['gdp00'].isnull(), 'gdp00'] = imputed_gdp00_values
df.loc[df['gdp16'].isnull(), 'gdp16'] = imputed_gdp16_values

# Check results
missing_gdp00_after = df['gdp00'].isnull().sum()
missing_gdp16_after = df['gdp16'].isnull().sum()

print(f"Missing values before imputation:")
print(f"gdp00: {missing_gdp00_before}")
print(f"gdp16: {missing_gdp16_before}")

print(f"\nMissing values after imputation:")
print(f"gdp00: {missing_gdp00_after}")
print(f"gdp16: {missing_gdp16_after}")

# Handle missing values - impute with median for continuous variables
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].median())

print(f"\nFinal dataset shape: {df.shape}")
print(f"Variables: {list(df.columns)}")

# 2. Univariate Significance Testing
print("\n" + "="*50)
print("2. UNIVARIATE SIGNIFICANCE TESTING")
print("="*50)

# Test each predictor individually
univariate_results = []

for predictor in potential_predictors:
    if predictor != 'gold16':
        # Remove rows where either variable is missing
        temp_df = df[[predictor, 'gold16']].dropna()
        
        if len(temp_df) > 10:  # Ensure sufficient data
            # Pearson correlation test for continuous variables
            if temp_df[predictor].nunique() > 10:  # Continuous variable
                try:
                    corr, p_value = stats.pearsonr(temp_df[predictor], temp_df['gold16'])
                    test_type = "Pearson"
                except:
                    continue
            else:  # Categorical/binary variable
                # Group by predictor and compare means
                groups = [group['gold16'].values for name, group in temp_df.groupby(predictor)]
                if len(groups) >= 2:
                    try:
                        f_stat, p_value = stats.f_oneway(*groups)
                        corr = np.sqrt(f_stat / (f_stat + len(temp_df) - len(groups)))
                        test_type = "ANOVA"
                    except:
                        continue
                else:
                    continue
              
            univariate_results.append({
                'predictor': predictor,
                'correlation': corr,
                'p_value': p_value,
                'test_type': test_type,
                'significant': p_value < 0.05,
                'n_obs': len(temp_df)
            })

# Display univariate results
if univariate_results:
    univariate_df = pd.DataFrame(univariate_results)
    univariate_df = univariate_df.sort_values('p_value')
    print("\nUnivariate Significance Results:")
    print(univariate_df.to_string(index=False))
    
    # Show top 20 most significant predictors
    print(f"\nTop 20 Most Significant Predictors:")
    top_predictors = univariate_df.head(20)
    for i, row in top_predictors.iterrows():
        stars = "***" if row['p_value'] < 0.001 else "**" if row['p_value'] < 0.01 else "*" if row['p_value'] < 0.05 else ""
        print(f"{row['predictor']:20} : r={row['correlation']:6.3f} p={row['p_value']:.2e} {stars}")
else:
    print("No significant results found.")

Data Preparation...
Countries with missing gdp16: ['Cuba', 'Syrian Arab Republic']
Countries with missing gdp00: ['Afghanistan']
Using predictors: ['gdp12', 'gdp08', 'gdp04', 'gdp00', 'gdp16', 'pop16', 'pop12', 'pop08', 'pop04', 'pop00', 'soviet', 'comm', 'muslim', 'oneparty']

Data for imputation shape: (108, 14)
Missing values before imputation:
gdp00: 1
gdp16: 2

KNN IMPUTATION RESULTS

GDP00 Imputation Results:
Afghanistan: 42,653 (imputed)

GDP16 Imputation Results:
Cuba: 62,427 (imputed)
Syrian Arab Republic: 51,636 (imputed)

UPDATING ORIGINAL COLUMNS
Missing values before imputation:
gdp00: 1
gdp16: 2

Missing values after imputation:
gdp00: 0
gdp16: 0

Final dataset shape: (108, 44)
Variables: ['country', 'country.code', 'gdp00', 'gdp04', 'gdp08', 'gdp12', 'gdp16', 'pop00', 'pop04', 'pop08', 'pop12', 'pop16', 'soviet', 'comm', 'muslim', 'oneparty', 'gold00', 'gold04', 'gold08', 'gold12', 'gold16', 'tot00', 'tot04', 'tot08', 'tot12', 'tot16', 'totgold00', 'totgold04', 'totgold0

In [3]:
import pandas as pd

# Sort by the target
df_sorted = df.sort_values(by='gold16', ascending=False)

# Keep all but the top 1 point
n_remove = 1
df_filtered = df_sorted.iloc[n_remove:]  # drops the top n_remove rows

# Optional: reset index
df_filtered = df_filtered.reset_index(drop=True)

# Drop unnecessary rows
df_upd = df_filtered.drop(columns=['country','country.code','oneparty','altitude','bmi','muslim','comm','soviet', 'totgold00', 'totgold04', 'totgold08', 'totgold12', 'totgold16', 
    'totmedals00', 'totmedals04', 'totmedals08', 'totmedals12', 'totmedals16','gold16'])
df_upd

,gdp00,gdp04,gdp08,gdp12,gdp16,pop00,pop04,pop08,pop12,pop16,...,tot04,tot08,tot12,tot16,athletes00,athletes04,athletes08,athletes12,athletes16,host
0,1647951.0,2398555,2890564,2662085,2650850.0,58893,59988,61807,63700,65596,...,30,49,65,67,310,264,305,530,360,1
1,1211347.0,1955347,4598206,8560547,11190993.0,1262645,1296075,1324655,1350695,1378665,...,63,100,91,70,271,384,599,375,392,1
2,259708.0,591017,1660844,2210257,1284728.0,146597,144067,142742,143202,144342,...,90,60,70,55,435,446,454,430,285,1
3,1949954.0,2819245,3752366,3543984,3477796.0,82212,82516,82110,80426,82349,...,49,41,44,42,422,441,420,383,418,1
4,4887520.0,4815149,5037908,6203213,4949273.0,126843,127761,128063,127629,126995,...,37,25,38,41,266,306,332,291,336,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,7938.0,14373,35596,28120,27572.0,2368,2263,2177,2034,1960,...,4,3,2,0,45,32,47,45,32,0
103,1370.0,2212,5140,6605,6813.0,4898,5105,5319,5607,6080,...,0,3,0,0,48,29,20,14,19,0
104,54790.0,85325,171001,209059,159049.0,31184,32831,34861,37566,40606,...,0,2,1,2,47,61,56,38,64,0
105,132339.0,135445,215840,257297,317748.0,6289,6809,7309,7911,8546,...,2,1,0,2,39,36,43,37,47,0


In [1]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

X = df_upd.values
Y = df_filtered['gold16']

# Standardize features before PCA (important!)
X_scaled = StandardScaler().fit_transform(X)

# Run PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Explained variance ratio
explained_variance = pca.explained_variance_ratio_
cumulative_variance = explained_variance.cumsum()

# Put into DataFrame for easy viewing
variance_df = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(explained_variance))],
    "Explained_Variance": explained_variance,
    "Cumulative_Variance": cumulative_variance
})

print(variance_df)

# Plot cumulative variance
plt.figure(figsize=(8,6))
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, marker='o')
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA - Cumulative Variance Explained")
plt.grid(True)
plt.show()

pca.components_

NameError: name 'df_upd' is not defined

In [5]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP, NegativeBinomialP
from sklearn.preprocessing import RobustScaler

# ---------- Train / test split ----------
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_scaled, Y.astype(int).values, df_filtered.index, test_size=0.2, random_state=42
)

# ---------- PCA (keep 6) ----------
pca = PCA(n_components=6, svd_solver='auto', random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

# Re-standardize the PCA scores (crucial to avoid overflow in exp())
scaler_pca = RobustScaler()
X_train_pca_s = scaler_pca.fit_transform(X_train_pca)
X_test_pca_s  = scaler_pca.transform(X_test_pca)

# Name the PC columns so statsmodels prints nice names
pc_cols = [f"PC{i+1}" for i in range(X_train_pca_s.shape[1])]
X_train_df = pd.DataFrame(X_train_pca_s, columns=pc_cols, index=np.asarray(idx_train))
X_test_df  = pd.DataFrame(X_test_pca_s,  columns=pc_cols, index=np.asarray(idx_test))

# Add intercept
X_train_df_const = sm.add_constant(X_train_df, has_constant='add')
X_test_df_const  = sm.add_constant(X_test_df,  has_constant='add')

X_train_df_const = X_train_df_const.clip(-3, 3)
X_test_df_const  = X_test_df_const.clip(-3, 3)

# ---------- Fit NB first to get sane starting params ----------
nb = NegativeBinomialP(y_train, X_train_df_const)
nb_res = nb.fit(maxiter=200, disp=False)

# Inflation part: use intercept-only for stability
exog_infl_train = np.ones((len(X_train_df_const), 1))
exog_infl_test  = np.ones((len(X_test_df_const), 1))

# Build start params: [count_params, infl_params, alpha]
start_params = np.concatenate([
    nb_res.params.values,               # k_count params (incl. const)
    np.zeros(exog_infl_train.shape[1]), # k_infl (here 1 intercept)
    [0.5]                               # alpha (dispersion)
])

# ---------- ZINB (NB2 variance) ----------
zinb = ZeroInflatedNegativeBinomialP(
    endog=y_train,
    exog=X_train_df_const,
    exog_infl=exog_infl_train,
    inflation='logit',
    p=1,  # NB2: Var = mu * (1 + alpha * mu)
)

# Compute the expected number of parameters
n_count = X_train_df_const.shape[1]
n_infl  = exog_infl_train.shape[1]
n_total = n_count + n_infl + 1  # +1 for alpha

# Try to reuse NB params if they match, else pad
nb_params = nb_res.params.values
if len(nb_params) < n_count:
    # pad with zeros if NB model dropped a col (rare)
    nb_params = np.concatenate([nb_params, np.zeros(n_count - len(nb_params))])

# Build the full vector
start_params = np.concatenate([
    nb_params[:n_count],
    np.zeros(n_infl),
    [0.5]
])

# Double-check shape
print("Expected total parameters:", n_total)
print("start_params length:", len(start_params))


zinb_res = zinb.fit(start_params=start_params, method='bfgs', maxiter=400, disp=False)
print(zinb_res.summary())

# ---------- Predict & evaluate ----------
y_pred = zinb_res.predict(exog=X_test_df_const, exog_infl=exog_infl_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print("\nTest Set Evaluation:")
print(f"MAE:  {mae:.3f}")
print(f"MSE:  {mse:.3f}")
print(f"RMSE: {rmse:.3f}")



Expected total parameters: 9
start_params length: 9
                     ZeroInflatedNegativeBinomialP Regression Results                    
Dep. Variable:                                 y   No. Observations:                   85
Model:             ZeroInflatedNegativeBinomialP   Df Residuals:                       78
Method:                                      MLE   Df Model:                            6
Date:                           Sun, 09 Nov 2025   Pseudo R-squ.:                  0.2578
Time:                                   00:17:12   Log-Likelihood:                -117.10
converged:                                  True   LL-Null:                       -157.78
Covariance Type:                       nonrobust   LLR p-value:                 1.872e-15
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
inflate_const   -17.1536   1300.240     -0.013      0.989   -256

C:\Users\M S I\anaconda3\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
C:\Users\M S I\anaconda3\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


In [6]:
# Round predictions to nearest integer (optional, since counts are discrete)
y_pred_rounded = y_pred.round().astype(int)

# Get country names for the test set
countries_test = df_filtered.loc[idx_test, 'country'].values

# Combine with actual values in a DataFrame
predictions_df = pd.DataFrame({
    'Country': countries_test,
    'Actual': y_test,
    'Predicted': y_pred_rounded,
})

print(predictions_df)

            Country  Actual  Predicted
76          Finland       0          0
10      Netherlands       8          4
4             Japan      12         15
99          Ireland       0          1
70           Kuwait       0          1
66           Uganda       0          1
30         Thailand       2          1
45          Bahrain       1          1
94          Morocco       0          1
11           Brazil       7          5
78          Eritrea       0          1
47      Puerto Rico       1          1
0    United Kingdom      27         26
79            Egypt       0          1
18           Canada       4          7
105          Israel       0          1
55             Fiji       1          1
77          Estonia       0          1
65          Tunisia       0          1
42       Azerbaijan       1          1
12            Spain       7          6
36      North Korea       2          1


In [9]:
# Save to a specific folder (Windows example)
predictions_df.to_excel(r"C:\Users\M S I\Downloads\goldpred.xlsx", index=False)